# Penguins Dataset - MLP

### [Penguins Dataset](https://seaborn.pydata.org/tutorial/introduction.html)

Author: [Kevin Thomas](mailto:ket189@pitt.edu)

License: MIT

## Install Libraries

In [ ]:
# conda activate prod
# conda install -c conda-forge pytorch torchvision torchaudio
# conda install numpy pandas matplotlib seaborn scikit-learn

## Import Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.feature_selection import f_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

## Seed

In [ ]:
SEED = 42
SEED

In [ ]:
torch.manual_seed(SEED)

## Parameters

In [ ]:
IN_FEATURES = 3
IN_FEATURES

In [ ]:
H1 = 16
H1

In [ ]:
H2 = 16
H2

In [ ]:
OUT_FEATURES = 3
OUT_FEATURES

In [ ]:
TEST_SIZE = 0.3
TEST_SIZE

In [ ]:
MODE = "classification"  # "classification" or "regression"
MODE

## Hyperparameters

In [ ]:
LEARNING_RATE = 0.01
LEARNING_RATE

In [ ]:
EPOCHS = 200
EPOCHS

In [ ]:
BATCH_SIZE = 16
BATCH_SIZE

In [ ]:
DROPOUT = 0.0
DROPOUT

## Load Dataset

In [ ]:
df = pd.read_csv('penguins-clean.csv')

In [ ]:
df.head()

## Clean Dataset

Map the three species to integers, keep the three continuous measurements, and standardize them so the network trains evenly.

In [ ]:
SPECIES_MAP = {'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2}
df['species_id'] = df['species'].map(SPECIES_MAP)
FEATURES = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
X = df[FEATURES].to_numpy(dtype=np.float32)
y = df['species_id'].to_numpy(dtype=np.int64)
X_MEAN = X.mean(axis=0)
X_STD = X.std(axis=0)
X = (X - X_MEAN) / X_STD
X.shape, y.shape

## Feature Statistical Significance (P-Values)

All three features are statistically significant. `flipper_length_mm` has the strongest separation between species.

In [ ]:
f_values, p_values = f_classif(X, y)
pd.DataFrame({'feature': FEATURES,
              'f_value': f_values,
              'p_value': p_values}).sort_values('p_value')

## Device

In [ ]:
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu")
DEVICE

## Create Model

In [ ]:
class Model(nn.Module):
    """
    A feedforward neural network with two hidden layers and optional dropout.
    """

    def __init__(self, in_features=IN_FEATURES, h1=H1, h2=H2,
                 out_features=OUT_FEATURES, dropout=DROPOUT):
        """
        Initialize the neural network layers.

        Parameters:
            in_features (int): Number of input features.
            h1 (int): Neurons in the first hidden layer.
            h2 (int): Neurons in the second hidden layer.
            out_features (int): Number of output features.
            dropout (float): Dropout rate (0.0 = no dropout).

        Returns:
            None
        """
        super(Model, self).__init__()
        self.fc1 = nn.Linear(in_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, out_features)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        """
        Run the forward pass of the network.

        Parameters:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output logits.
        """
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.out(x)

## Instantiate Model

In [ ]:
torch.manual_seed(SEED)
model = Model().to(DEVICE)
model

## Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train.shape, X_test.shape

## Create Custom Dataset (Memory Efficient)

In [ ]:
class NumpyDataset(torch.utils.data.Dataset):
    """
    A memory-efficient dataset that stores data as NumPy arrays.
    """

    def __init__(self, X, y, device, mode="classification"):
        """
        Initialize the dataset with NumPy arrays.

        Parameters:
            X (np.ndarray): Feature array.
            y (np.ndarray): Label array.
            device (torch.device): Device to move tensors to.
            mode (str): "classification" or "regression".

        Returns:
            None
        """
        self.X = X
        self.y = y
        self.device = device
        self.mode = mode

    def __len__(self):
        """
        Return the number of samples in the dataset.

        Parameters:
            None

        Returns:
            int: Number of samples.
        """
        return len(self.X)

    def __getitem__(self, idx):
        """
        Return a single sample as tensors on the device.

        Parameters:
            idx (int): Index of the sample.

        Returns:
            tuple: Feature and label tensors.
        """
        X_tensor = torch.tensor(self.X[idx]).float().to(self.device)
        if self.mode == "classification":
            y_tensor = torch.tensor(self.y[idx]).long().to(self.device)
        else:
            y_tensor = torch.tensor(self.y[idx]).float().to(self.device)
        return X_tensor, y_tensor

## Create DataLoaders

In [ ]:
train_dataset = NumpyDataset(X_train, y_train, DEVICE, MODE)
test_dataset = NumpyDataset(X_test, y_test, DEVICE, MODE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
len(train_loader), len(test_loader)

## Create Loss Function & Optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Train Model

### Functions

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """
    Train the model for a single epoch.

    Parameters:
        model (nn.Module): The model to train.
        loader (DataLoader): Training data loader.
        loss_fn (nn.Module): Loss function.
        optimizer (torch.optim.Optimizer): Parameter update rule.

    Returns:
        float: Mean training loss for the epoch.
    """
    model.train()
    total = 0.0
    for x, y in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    """
    Evaluate the model over a loader.

    Parameters:
        model (nn.Module): The model to evaluate.
        loader (DataLoader): Evaluation data loader.
        loss_fn (nn.Module): Loss function.

    Returns:
        tuple: Mean loss and accuracy.
    """
    model.eval()
    total, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            logits = model(x)
            total += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total / n, correct / n

### Training Loop

In [ ]:
history = {'loss': [], 'accuracy': []}
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
    history['loss'].append(test_loss)
    history['accuracy'].append(test_accuracy)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1:3d} | loss {test_loss:.4f} | accuracy {test_accuracy:.4f}")

### Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['loss'])
axes[0].set_title('Test loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(history['accuracy'])
axes[1].set_title('Test accuracy')
axes[1].set_xlabel('Epoch')
fig.tight_layout()
plt.show()

## Evaluate Model

In [ ]:
_, accuracy = evaluate(model, test_loader, loss_fn)
print(f"Test accuracy: {accuracy:.4f}")

In [ ]:
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        all_preds.append(model(x).argmax(1).cpu().numpy())
        all_labels.append(y.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=list(SPECIES_MAP)))

## Save Model

In [ ]:
torch.save(model.state_dict(), 'mlp_penguins.pt')
print('saved mlp_penguins.pt')

## Load Model

In [ ]:
loaded_model = Model().to(DEVICE)
loaded_model.load_state_dict(torch.load('mlp_penguins.pt', map_location=DEVICE))
loaded_model.eval()
print('loaded mlp_penguins.pt')

## Inference

### Function

In [ ]:
def predict(model, features):
    """
    Predict the species for one standardized measurement.

    Parameters:
        model (nn.Module): Trained model.
        features (list): Raw bill length, bill depth, and flipper length.

    Returns:
        tuple: Predicted species name and confidence.
    """
    model.eval()
    scaled = (np.array(features, dtype=np.float32) - X_MEAN) / X_STD
    with torch.no_grad():
        X_new = torch.tensor(scaled).float().to(DEVICE).unsqueeze(0)
        probs = torch.softmax(model(X_new), dim=1)
        confidence, predicted = probs.max(dim=1)
    return list(SPECIES_MAP)[predicted.item()], confidence.item()

### Run Inference

In [ ]:
species, confidence = predict(loaded_model, [47.5, 15.0, 217.0])
print(f"Predicted species: {species} ({confidence:.4f})")